In [1]:
# Import Libraries

import pandas as pd

In [2]:
# Load Cleaned Datasets

logon_data = pd.read_csv("../dataset/logon_cleaned.csv")
device_data = pd.read_csv("../dataset/device_cleaned.csv")
email_data = pd.read_csv("../dataset/email_cleaned.csv")
file_data = pd.read_csv("../dataset/file_cleaned.csv")
psychometric_data = pd.read_csv("../dataset/psychometric_cleaned.csv")

In [3]:
# Convert Date Column

logon_data["date"] = pd.to_datetime(logon_data["date"])
device_data["date"] = pd.to_datetime(device_data["date"])
email_data["date"] = pd.to_datetime(email_data["date"])
file_data["date"] = pd.to_datetime(file_data["date"])

In [4]:
# Create Login Count Feature

login_count = logon_data.groupby("user").size().reset_index(name="login_count")

In [5]:
# View Login Count

login_count.head()

,user,login_count
0,AAE0190,692
1,AAF0535,328
2,AAF0791,692
3,AAL0706,692
4,AAM0658,458


In [6]:
# Create Logoff Count Feature

logoff_count = (
    logon_data[logon_data["activity"] == "Logoff"]
    .groupby("user")
    .size()
    .reset_index(name="logoff_count")
)

In [7]:
# View Logoff Count

logoff_count.head()

,user,logoff_count
0,AAE0190,346
1,AAF0535,164
2,AAF0791,346
3,AAL0706,346
4,AAM0658,229


In [8]:
# Create USB Connect Count

usb_connect_count = (
    device_data[device_data["activity"] == "Connect"]
    .groupby("user")
    .size()
    .reset_index(name="usb_connect_count")
)

In [9]:
# View USB Connect Count

usb_connect_count.head()

,user,usb_connect_count
0,AAF0535,346
1,AAM0658,7
2,ABC0174,642
3,AHD0848,150
4,AHM0410,394


In [10]:
# Create USB Disconnect Count

usb_disconnect_count = (
    device_data[device_data["activity"] == "Disconnect"]
    .groupby("user")
    .size()
    .reset_index(name="usb_disconnect_count")
)

In [11]:
# View USB Disconnect Count

usb_disconnect_count.head()

,user,usb_disconnect_count
0,AAF0535,342
1,AAM0658,6
2,ABC0174,634
3,AHD0848,149
4,AHM0410,388


In [12]:
# Create Email Count

email_count = (
    email_data.groupby("user")
    .size()
    .reset_index(name="email_count")
)

In [13]:
# View Email Count

email_count.head()

,user,email_count
0,AAE0190,4711
1,AAF0535,480
2,AAF0791,3012
3,AAL0706,336
4,AAM0658,659


In [14]:
# Create File Activity Count

file_activity_count = (
    file_data.groupby("user")
    .size()
    .reset_index(name="file_activity_count")
)

In [15]:
# View File Activity Count

file_activity_count.head()

,user,file_activity_count
0,AAF0535,357
1,AAM0658,31
2,ABC0174,589
3,AHD0848,199
4,AHM0410,2198


In [16]:
# Create Unique PC Count

unique_pc_count = (
    logon_data.groupby("user")["pc"]
    .nunique()
    .reset_index(name="unique_pc_count")
)

In [17]:
# View Unique PC Count

unique_pc_count.head()

,user,unique_pc_count
0,AAE0190,1
1,AAF0535,1
2,AAF0791,1
3,AAL0706,1
4,AAM0658,1


In [18]:
# Create Working Hour

logon_data["hour"] = logon_data["date"].dt.hour

In [19]:
# Create After Hours Login Count

after_hours_login = (
    logon_data[(logon_data["hour"] < 8) | (logon_data["hour"] > 18)]
    .groupby("user")
    .size()
    .reset_index(name="after_hours_login")
)

In [20]:
# View After Hours Login Count

after_hours_login.head()

,user,after_hours_login
0,AAL0706,346
1,AAM0658,234
2,AAN0823,346
3,AAS0442,22
4,AAV0450,159


In [21]:
# Create Weekend Activity Count

logon_data["day"] = logon_data["date"].dt.day_name()

weekend_activity = (
    logon_data[logon_data["day"].isin(["Saturday", "Sunday"])]
    .groupby("user")
    .size()
    .reset_index(name="weekend_activity")
)

In [22]:
# View Weekend Activity Count

weekend_activity.head()

,user,weekend_activity
0,AAM0658,2
1,ABC0174,126
2,ACV0812,13
3,AHC0142,100
4,AHD0848,87


In [27]:
# Create Attachment Count

attachment_count = (
    email_data[email_data["attachments"] > 0]
    .groupby("user")
    .size()
    .reset_index(name="attachment_count")
)

In [28]:
# View Attachment Count

attachment_count.head()

,user,attachment_count
0,AAE0190,953
1,AAF0535,187
2,AAL0706,71
3,AAM0658,275
4,AAN0823,78


In [29]:
# Merge All Features

features = login_count.merge(logoff_count, on="user", how="outer")

features = features.merge(usb_connect_count, on="user", how="outer")
features = features.merge(usb_disconnect_count, on="user", how="outer")
features = features.merge(email_count, on="user", how="outer")
features = features.merge(file_activity_count, on="user", how="outer")
features = features.merge(unique_pc_count, on="user", how="outer")
features = features.merge(after_hours_login, on="user", how="outer")
features = features.merge(weekend_activity, on="user", how="outer")
features = features.merge(attachment_count, on="user", how="outer")

In [30]:
# Replace Missing Values

features = features.fillna(0)

In [31]:
# View Final Feature Dataset

features.head()

,user,login_count,logoff_count,usb_connect_count,usb_disconnect_count,email_count,file_activity_count,unique_pc_count,after_hours_login,weekend_activity,attachment_count
0,AAE0190,692,346,0.0,0.0,4711,0.0,1,0.0,0.0,953.0
1,AAF0535,328,164,346.0,342.0,480,357.0,1,0.0,0.0,187.0
2,AAF0791,692,346,0.0,0.0,3012,0.0,1,0.0,0.0,0.0
3,AAL0706,692,346,0.0,0.0,336,0.0,1,346.0,0.0,71.0
4,AAM0658,458,229,7.0,6.0,659,31.0,1,234.0,2.0,275.0


In [32]:
# Check Dataset Shape

features.shape

(1000, 11)

In [33]:
# Save Final Feature Dataset

features.to_csv("../dataset/final_features.csv", index=False)

In [34]:
# Check Saved Dataset

import os

os.path.exists("../dataset/final_features.csv")

True